In [ ]:
import pandas as pd

oportunidad = pd.read_excel("../data/inputs/oportunidad.xlsx")
oportunidad.head()

## Contexto: estrategia FDS anterior

Resultado de la estrategia de precios del fin de semana anterior (jun 2026), usado como referencia para la nueva estrategia.

In [ ]:
analisis_path = "../data/utils/analisis_estrategia_pricing_fds_jun2026.xlsx"

detalle_skus = pd.read_excel(analisis_path, sheet_name="Detalle SKUs")
resumen_departamento = pd.read_excel(analisis_path, sheet_name="Resumen Departamento")
top_uplift = pd.read_excel(analisis_path, sheet_name="Top Uplift")
sin_venta_fds = pd.read_excel(analisis_path, sheet_name="Sin Venta FDS")

detalle_skus.head()

## Estrategia de ofertas FDS

Construye la tabla de SKUs priorizados con mecánica recomendada y margen simulado post-oferta, con margen mínimo del 22%.

In [ ]:
MARGEN_MIN = 22.0

# Limpiar oportunidad: quitar filas de resumen/filtros al final del export y parsear SKU
uni = oportunidad.dropna(subset=["STORE_ID"]).copy()
uni = uni[uni["SKU + Nombre"].astype(str).str.match(r"^\d+\s*-\s*.+")].copy()

uni[["SKU", "Nombre"]] = uni["SKU + Nombre"].str.split(" - ", n=1, expand=True)
uni["SKU"] = uni["SKU"].astype(int)
uni["STORE_ID"] = uni["STORE_ID"].astype(int)
uni.head()

### Excluir Black list y descuentos comerciales vigentes

`Descuentos comerciales.xlsx` trae dos pestañas por SKU + Tienda (`ItemId` + `AreaId`): `Black list` (exclusion fija) y `DEscuentos comerciales` (calendario de campañas, `FechaInicio`/`FechaFin`). Se excluye del universo cualquier SKU en Black list, o con un descuento comercial cuyo rango de fechas se cruce con el fin de semana objetivo (viernes 3 - domingo 5 jul 2026), para no duplicar mecánicas sobre un SKU que ya tiene oferta comprometida.

In [ ]:
WEEKEND_INICIO = pd.Timestamp("2026-07-03")
WEEKEND_FIN = pd.Timestamp("2026-07-05")

descuentos_path = "../data/inputs/Descuentos comerciales.xlsx"

black_list = pd.read_excel(descuentos_path, sheet_name="Black list")
black_pares = set(zip(black_list["ItemId"].astype(int), black_list["AreaId"].astype(int)))

comerciales = pd.read_excel(descuentos_path, sheet_name="DEscuentos comerciales")
comerciales["ItemId"] = pd.to_numeric(comerciales["ItemId"], errors="coerce")
comerciales["AreaId"] = pd.to_numeric(comerciales["AreaId"], errors="coerce")
comerciales = comerciales.dropna(subset=["ItemId", "AreaId", "FechaInicio", "FechaFin"])
comerciales["ItemId"] = comerciales["ItemId"].astype(int)
comerciales["AreaId"] = comerciales["AreaId"].astype(int)

vigentes = comerciales[
    (comerciales["FechaInicio"] <= WEEKEND_FIN) & (comerciales["FechaFin"] >= WEEKEND_INICIO)
]
comerciales_pares = set(zip(vigentes["ItemId"], vigentes["AreaId"]))

excluir_pares = black_pares | comerciales_pares
print(f"Pares (SKU, Tienda) en Black list: {len(black_pares)}")
print(f"Pares (SKU, Tienda) con descuento comercial vigente ese finde: {len(comerciales_pares)}")

antes = len(uni)
mask_excluir = uni.set_index(["SKU", "STORE_ID"]).index.isin(excluir_pares)
uni = uni[~mask_excluir].copy()
print(f"Excluidos por Black list / descuento comercial vigente: {antes - len(uni)}")
print(f"Universo restante: {len(uni)}")

## Catalogo SKU / Departamento / Categoria (Snowflake)

Reemplaza el cruce contra `analisis_estrategia_pricing_fds_jun2026.xlsx` (solo 110 SKUs, 3.5% de cobertura) por el catalogo real. Requiere `.env` con `SNOWFLAKE_USER`, `SNOWFLAKE_WAREHOUSE` y `SNOWFLAKE_ROLE` llenos. Autenticacion SSO de Google: al ejecutar la celda se abre el navegador.

In [ ]:
from snowflake_conn import get_connection

conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE()")
print(cur.fetchone())

In [ ]:
cur.execute("""
    SELECT DISTINCT SKU, DEPARTMENT, CATEGORY
    FROM MX_JUSTO_PROD.SANDBOX.FULL_MASTER_CATALOG
""")
columnas = [c[0] for c in cur.description]
catalogo = pd.DataFrame(cur.fetchall(), columns=columnas)
catalogo = catalogo.rename(columns={"DEPARTMENT": "Departamento", "CATEGORY": "Categoria"})

# SKU puede venir como VARCHAR sucio; forzar a entero y descartar lo no numerico
catalogo["SKU"] = pd.to_numeric(catalogo["SKU"], errors="coerce")
catalogo = catalogo.dropna(subset=["SKU"])
catalogo["SKU"] = catalogo["SKU"].astype(int)

# Un SKU podria tener mas de una combinacion Departamento/Categoria (data quality) - nos quedamos con la primera
dup = catalogo["SKU"].duplicated().sum()
print(f"SKUs con mas de una combinacion Departamento/Categoria: {dup}")
catalogo = catalogo.drop_duplicates(subset="SKU", keep="first")

print(f"SKUs en catalogo: {len(catalogo)}")
catalogo.head()

In [ ]:
uni = uni.merge(catalogo, on="SKU", how="left")

cobertura = uni["Departamento"].notna().sum()
print(f"Cobertura Departamento/Categoria: {cobertura}/{len(uni)} filas ({cobertura / len(uni) * 100:.1f}%)")
uni.head()

In [ ]:
# Validar que MARGEN = margen sobre precio neto de IVA e IEPS
# (PRECIO JUSTO viene con impuestos incluidos)
precio_neto_check = uni["PRECIO JUSTO"] / (1 + uni["IVA"] / 100) / (1 + uni["IEPS"] / 100)
margen_check = (precio_neto_check - uni["COSTO"]) / precio_neto_check * 100
diff = (margen_check - uni["MARGEN"]).abs()
print(f"Diferencia vs columna MARGEN: media={diff.mean():.4f} pts, "
      f"filas con diff>0.5pts={(diff > 0.5).sum()}/{len(uni)}")

In [ ]:
# Filtros de exclusion: margen base < 22%, o inelastico/sin datos de elasticidad
antes = len(uni)
elegibles = uni[uni["MARGEN"] >= MARGEN_MIN]
excl_margen = antes - len(elegibles)

antes = len(elegibles)
elegibles = elegibles[~elegibles["TAG ELAS"].isin(["INELASTICO", "SIN DATOS"])].copy()
excl_elas = antes - len(elegibles)

print(f"Excluidos por margen base < {MARGEN_MIN}%: {excl_margen}")
print(f"Excluidos por TAG_ELAS inelastico/sin datos: {excl_elas}")
print(f"SKUs elegibles: {len(elegibles)}")

In [ ]:
TOLERANCIA_CASCADA = 2.0  # puntos % max. que se sacrifican para usar una mecanica con nombre


def simular_oferta(precio, costo, ieps, iva):
    """Calcula el descuento maximo exacto (d_max) que deja el margen (neto de
    IVA/IEPS) justo en MARGEN_MIN - esa es la unica fuente de verdad, nunca se
    ofrece menos descuento del que el margen permite.

    Se usa una mecanica con nombre (2x1/3x2/4x3/5x4/6x5, la familia "paga N-1
    lleva N") solo si su descuento real esta a <= TOLERANCIA_CASCADA puntos de
    d_max (para no sacrificar atractivo solo por usar un nombre redondo). Si
    ninguna mecanica con nombre esta lo bastante cerca, se usa el descuento
    exacto, presentado segun la "regla del 100" de pricing: SPON (%) para
    precios < $100 (el % se percibe mas grande), BNSDP ($) para precios >=
    $100 (el monto en pesos pega mas fuerte).
    """

    def margen_neto(precio_final):
        precio_neto = precio_final / (1 + iva / 100) / (1 + ieps / 100)
        return (precio_neto - costo) / precio_neto * 100

    precio_neto_min = costo / (1 - MARGEN_MIN / 100)
    precio_final_min = precio_neto_min * (1 + iva / 100) * (1 + ieps / 100)
    d_max = max((1 - precio_final_min / precio) * 100, 0)

    if d_max <= 0:
        return "Sin oferta viable", precio, margen_neto(precio), "Sin oferta viable", round(d_max, 2)

    for nombre, ratio in [("2x1", 50.0), ("3x2", 100 / 3), ("4x3", 25.0), ("5x4", 20.0), ("6x5", 100 / 6)]:
        if ratio <= d_max:
            if (d_max - ratio) <= TOLERANCIA_CASCADA:
                pe = precio * (1 - ratio / 100)
                return nombre, pe, margen_neto(pe), "Cascada (paga N-1 lleva N)", round(d_max, 2)
            break  # esta mecanica deja mas de 2 pts sin usar; ninguna mas chica sera mejor

    # Descuento exacto, redondeado a multiplos de 5 (o al entero si el espacio es <5)
    d = (int(d_max) // 5) * 5 if d_max >= 5 else int(d_max)
    d = max(d, 0)
    pe = precio * (1 - d / 100)
    margen = margen_neto(pe)

    if precio >= 100:
        ahorro = round(precio - pe)
        return f"BNSDP ahorra ${ahorro}", pe, margen, "BNSDP (pesos, precio >= $100)", round(d_max, 2)
    return f"SPON {d}%", pe, margen, "SPON (%, precio < $100)", round(d_max, 2)


resultados = elegibles.apply(
    lambda r: simular_oferta(r["PRECIO JUSTO"], r["COSTO"], r["IEPS"], r["IVA"]), axis=1
)
elegibles["Mecanica recomendada"] = resultados.str[0]
elegibles["Precio oferta efectivo"] = resultados.str[1].round(2)
elegibles["Margen simulado post-oferta %"] = resultados.str[2].round(2)
elegibles["Tipo mecanica"] = resultados.str[3]
elegibles["Descuento maximo posible %"] = resultados.str[4]

antes = len(elegibles)
elegibles = elegibles[elegibles["Mecanica recomendada"] != "Sin oferta viable"]
print(f"Excluidos por no encontrar oferta viable: {antes - len(elegibles)}")

### Deteccion "Tema Mundial"

No hay merchandising con licencia FIFA en este catalogo (0 matches para "mundial", "fútbol", "FIFA", "selección"), asi que se usa un proxy: categorias tipicas de "ver el partido" (cerveza, refrescos, botanas saladas, salsas picantes/dip). Revisado a mano para evitar falsos positivos (ej. se descarto "cacahuate" porque en este catalogo solo matcheaba crema de cacahuate/yogurt, no cacahuates como botana).

In [ ]:
import re

PATRON_MUNDIAL = re.compile(
    r"\b(?:"
    r"cerveza|refresco|"
    r"papas?\s+(?:fritas?|pringles|ruffles)|totopos?|nachos?|chicharr\w*|"
    r"palomitas|botanas?|"
    r"habanero|valentina|cholula|clamato|b[uú]falo"
    r")\b",
    re.IGNORECASE,
)
elegibles["Tema Mundial"] = elegibles["Nombre"].str.contains(PATRON_MUNDIAL, na=False)
print(f"SKUs marcados como afines al Mundial: {elegibles['Tema Mundial'].sum()}")

In [ ]:
# Dia sugerido: domingo para despensa (cobertura parcial de Departamento),
# viernes para alta rotacion, sabado como dia insignia por defecto ("mayor impacto")
def dia_sugerido(row):
    if row["Departamento"] == "Despensa":
        return "Domingo"
    if row["TAG_ROTACION"] == "Alta rotacion":
        return "Viernes"
    return "Sabado"


elegibles["Dia sugerido"] = elegibles.apply(dia_sugerido, axis=1)

# Orden: TAG_ELAS (mejor primero), luego TAG_ROTACION, luego margen simulado descendente
orden_elas = {"ALTAMENTE ELASTICO": 0, "ELASTICO": 1, "POCO ELASTICO": 2}
orden_rot = {"Alta rotacion": 0, "Rotacion normal": 1, "Baja rotacion": 2, "Sin rotacion": 3}

elegibles["_orden_elas"] = elegibles["TAG ELAS"].map(orden_elas)
elegibles["_orden_rot"] = elegibles["TAG_ROTACION"].map(orden_rot)

elegibles = elegibles.sort_values(
    by=["_orden_elas", "_orden_rot", "Margen simulado post-oferta %"],
    ascending=[True, True, False],
)

estrategia_fds = elegibles.rename(
    columns={"MARGEN": "Margen base %", "TAG ELAS": "TAG_ELAS"}
)[
    [
        "STORE_ID", "SKU", "Nombre", "Departamento", "Categoria",
        "Margen base %", "TAG_ELAS", "TAG_ROTACION",
        "Tipo mecanica", "Mecanica recomendada", "Descuento maximo posible %",
        "Margen simulado post-oferta %",
        "Precio oferta efectivo", "Dia sugerido", "Tema Mundial",
    ]
]

estrategia_fds.to_excel("../data/output/estrategia_fds.xlsx", index=False)

print("=== RESUMEN ===")
print(f"Total SKUs incluidos: {len(estrategia_fds)}")
print()
print("Distribucion por tipo de mecanica:")
print(estrategia_fds["Tipo mecanica"].value_counts())
print()
print("Distribucion por mecanica:")
print(estrategia_fds["Mecanica recomendada"].value_counts())
print()
print("Distribucion por dia:")
print(estrategia_fds["Dia sugerido"].value_counts())
print()
print(f"SKUs Tema Mundial: {estrategia_fds['Tema Mundial'].sum()}")
print(estrategia_fds[estrategia_fds["Tema Mundial"]]["Dia sugerido"].value_counts())